In [ ]:
import sys
import os
import pandas as pd
import numpy as np
from pycox.models.cox import CoxPH
from pycox.evaluation import EvalSurv
import matplotlib.pyplot as plt
import shap
import scipy.integrate
scipy.integrate.simps = scipy.integrate.simpson



from itertools import product
import torch

sys.path.append(os.path.abspath("../../"))
from src.dataset.generate_dataset import TorchPreprocessing
from src.utils.kapplan_meir import k_m_cox
from src.dataset.DataSet import SurvivalDataSet
from src.utils.Preprocessing import Preprocessor
from src.utils.ConvertTextToCsv import TextToCsv
from src.dataset.split_data import split_data_Train_Val_Test
from src.utils.set_seed import set_seed
from src.utils.cox_models import *
from sklearn.utils import resample


import numpy as np
import torch
import torchtuples as tt

%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
set_seed(42)

In [ ]:
pp = Preprocessor()

In [ ]:
df_clinical_data = pd.read_csv("../../data/raw/brca_tcga_pub2015_clinical_data.tsv", sep='\t')
df_clinical_data = pp.clean_columns_dataset(df_clinical_data)
list_df = pp.total_type_len_type_cancer(df_clinical_data)
df_clinical_data["Tumor-Cancer"] = list_df
df_clinical_data["Tumor-Cancer"].unique()

df_mRNA_raw_data = TextToCsv("../../data/raw/data_mrna_seq_v2_rsem.txt")


In [ ]:
df_merged = TorchPreprocessing(df_mRNA_raw_data,df_clinical_data, 23360).get_comparation_df()
 
comparation_df = df_merged.loc[
    df_merged["Tumor-Cancer"].isin(["Luminal A", "Luminal B", "TNBC", "HER2-enriched"]),
]

comparation_df["Tumor-Cancer"].unique()

print(f"Samples: {comparation_df.shape[0]}, Genes: {comparation_df.shape[1]}")

In [ ]:
zero_reduced_df =  comparation_df.drop(["Sample ID"], axis=1)
results_df, desing, expr = pp.initialize_limma(zero_reduced_df, column="Tumor-Cancer", column_event="Overall Survival (Months)", column_status="Overall Survival Status")

In [ ]:
N_GENES = 1000
top_genes_limma = results_df.sort_values("pvalue").index[:N_GENES].tolist()

In [ ]:
torch_preprocessing = TorchPreprocessing(df_mRNA_raw_data, df_clinical_data, 23360)
torch_preprocessing.genes_expression = top_genes_limma
genes_expression = top_genes_limma
X_scaled, durations, events, scaler, sample_ids = torch_preprocessing.get_data_set(60, return_ids=True)


In [ ]:
surival_data_set  = SurvivalDataSet(X_scaled, durations, events)

In [ ]:
import json
from src.dataset.split_data import split_dataset_by_ids

# Load the canonical split created by limma.ipynb so the NN uses the SAME patients.
split = json.load(open("splits.json"))
set_seed()
(train_X, train_durations, train_events), \
(val_X, val_durations, val_events), \
(test_X, test_durations, test_events) = split_dataset_by_ids(
    surival_data_set, sample_ids,
    split["train_ids"], split["val_ids"], split["test_ids"]
)
print(f"Train: {train_X.shape[0]}, Val: {val_X.shape[0]}, Test: {test_X.shape[0]}")

In [ ]:
in_features = train_X.shape[1]
train_X_np = train_X.cpu().numpy()
train_durations_np = train_durations.cpu().numpy()
train_events_np = train_events.cpu().numpy()
test_X_np = test_X

train_X_np.shape, train_durations_np.shape, train_events_np.shape

In [ ]:
set_seed()
N_BOOTSTRAP = 10
gene_shap_records = {gene : [] for gene in genes_expression}

event_idx = np.where(train_events_np == 1)[0]
no_event_idx = np.where(train_events_np == 0)[0]


## DeepSurv Bootstrap

In [ ]:
for i in range(N_BOOTSTRAP):
    idx_ev = resample(event_idx, random_state=i, replace=True)
    idx_no = resample(no_event_idx, random_state=i, replace=True)
    idx = np.concatenate([idx_ev, idx_no])
    
    X_boot = train_X_np[idx]
    dur_boot = train_durations[idx]
    ev_boot = train_events_np[idx]
    
    ev_boot_idx = np.where(ev_boot == 1)[0]
    no_ev_boot_idx = np.where(ev_boot == 0)[0]
    
    val_ev = resample(ev_boot_idx,    n_samples=max(3, int(0.15*len(ev_boot_idx))),    random_state=i, replace=False)
    val_no = resample(no_ev_boot_idx, n_samples=max(5, int(0.15*len(no_ev_boot_idx))), random_state=i, replace=False)
    val_idx = np.concatenate([val_ev, val_no])
    train_idx = np.setdiff1d(np.arange(len(X_boot)), val_idx)
    
    
    X_tr, X_val = X_boot[train_idx], X_boot[val_idx]
    dur_tr, dur_val = dur_boot[train_idx], dur_boot[val_idx]
    ev_tr,  ev_val = ev_boot[train_idx], ev_boot[val_idx]
    
    net_MLPVanilla = tt.torchtuples.practical.MLPVanilla(
        in_features=in_features,
        num_nodes=[16, 16],
        dropout=0.2,
        batch_norm=True,
        output_bias=False,
        out_features=1
    )
    
    #DeepSurv
    model_boot = CoxPH(
        net_MLPVanilla,
        tt.torchtuples.optim.Adam(
            lr=0.0005,
            weight_decay=0.001 # type: ignore
        )
    )
    model_boot.fit(
        X_tr,
        (dur_tr, ev_tr),
        batch_size=256,
        epochs=200,
        verbose=False,
        callbacks=[tt.torchtuples.callbacks.EarlyStopping(patience=10)],
        val_data=(X_val, (dur_val, ev_val))
    )
    
    def predict_fn(x):
        x_tensor = torch.tensor(x, dtype=torch.float32)
        with torch.no_grad():
            out = model_boot.net(x_tensor)
        return out.cpu().numpy().reshape(-1)


    preds_check = predict_fn(X_boot[:5])
    if np.isnan(preds_check).any():
        print(f"  → Bootstrap {i+1} divergió, saltando")
        continue

    background = shap.sample(X_boot, 50)
    explainer = shap.explainers.Permutation(
        predict_fn, background,
        feature_names=genes_expression,
        max_evals=2001
    )
    shap_values = explainer(test_X_np)
    mean_abs = np.abs(shap_values.values).mean(axis=0)

    for j, gene in enumerate(genes_expression):
        gene_shap_records[gene].append(mean_abs[j])

    print(f"Bootstrap {i+1}/{N_BOOTSTRAP} OK")
    
    
    

### CoxNex DeppSurv

In [ ]:
for i in range(N_BOOTSTRAP):
    idx_ev = resample(event_idx, random_state=i, replace=True)
    idx_no = resample(no_event_idx, random_state=i, replace=True)
    idx = np.concatenate([idx_ev, idx_no])
    
    X_boot = train_X_np[idx]
    dur_boot = train_durations[idx]
    ev_boot = train_events_np[idx]
    
    ev_boot_idx = np.where(ev_boot == 1)[0]
    no_ev_boot_idx = np.where(ev_boot == 0)[0]
    
    val_ev = resample(ev_boot_idx,    n_samples=max(3, int(0.15*len(ev_boot_idx))),    random_state=i, replace=False)
    val_no = resample(no_ev_boot_idx, n_samples=max(5, int(0.15*len(no_ev_boot_idx))), random_state=i, replace=False)
    val_idx = np.concatenate([val_ev, val_no])
    train_idx = np.setdiff1d(np.arange(len(X_boot)), val_idx)
    
    
    X_tr, X_val = X_boot[train_idx], X_boot[val_idx]
    dur_tr, dur_val = dur_boot[train_idx], dur_boot[val_idx]
    ev_tr,  ev_val = ev_boot[train_idx], ev_boot[val_idx]
